# Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [20]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
whisper-large-v3
openai/gpt-oss-20b
allam-2-7b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
groq/compound
openai/gpt-oss-120b
qwen/qwen3.6-27b
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m


In [2]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

C:\Users\sangr\AppData\Local\Temp\ipykernel_1460\2433114369.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\sangr\Videos\AIML\openai-basic\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

In [6]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the re

In [7]:
#Divide our Docuemnts into chunks dcouments
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [8]:
documents

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the re

In [9]:
# Embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3223.28it/s]


In [10]:
# vector db
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [11]:
## Query From a vector db
query="LangSmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

"LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get"

In [23]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0
)

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Retriever
retriever = vectorstoredb.as_retriever()

# 2. Prompt
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question based only on the following context.

    Context:
    {context}

    Question:
    {input}
    """
)

# 3. Document/LLM chain
document_chain = prompt | llm | StrOutputParser()

# 4. Retrieval chain
retrieval_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough() #quition becomes input(for this case, what is machine learning)
    }
    | document_chain
)

# 5. Ask question
response = retrieval_chain.invoke(
    "LangSmith has two usage limits: total traces and extended"
)

print(response)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "LangSmith has two usage limits: total traces and extended"
   - **Context:** Four documents from LangSmith documentation about observability, setup, integrations, tracing, monitoring, etc.
   - **Task:** Answer the question based *only* on the provided context.

2.  **Scan Context for Keywords:**
   - Keywords: "usage limits", "total traces", "extended", "limits", "pricing", "retention"
   - Looking through the context:
     - Doc 1: Mentions "LangSmith Observability provides full visibility...", "Traces are the record...", "works with many frameworks..."
     - Doc 2: Mentions "For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing."
     - Doc 3: Mentions "evaluate against...", "works with many frameworks...", "Create an account..."
     - Doc 4: Mentions "Copy the key...", "Set up tracing...", "Investigate and monitor...", 



RAG pipeline

User Question
      ↓
   Retriever
       |
    vectorDB
      ↓
Relevant Documents
      ↓
    Context
      ↓
     Prompt
      ↓
      LLM
      ↓
    Answer

For example:

question = "What is deep learning?"

response = retrieval_chain.invoke(question)

print(response)

The retriever searches your FAISS database:

"What is deep learning?"
          ↓
       FAISS
          ↓
Relevant documents

Those documents become {context} in your prompt:

Answer the question based only on the context.

Context:
[retrieved documents]

Question:What is deep learning?
[Quistion becomes {input}]

Then the LLM generates the answer.